### Auto Loader File Detection Modes

Auto Loader supports two modes for detecting new files: **directory listing* and **file notification.** 

You can switch file discovery modes across stream restarts and still obtain exactly-once data processing guarantees.

#### Directory Listing Mode

In directory listing mode, **Auto Loader identifies new files by listing the input directory.** Directory listing mode allows you to quickly start Auto Loader streams without any permission configurations other than access to your data on cloud storage.

In Databricks Runtime 9.1 and above, Auto Loader can automatically detect whether files are arriving with lexical ordering to your cloud storage and significantly reduce the amount of API calls needed to detect new files. See [Auto Loader streams with directory listing mode](https://docs.databricks.com/aws/en/ingestion/cloud-object-storage/auto-loader/directory-listing-mode) for more details.

#### File Notification Mode

File notification mode leverages file notification and queue services in your cloud infrastructure account. Auto Loader can automatically set up a notification service and queue service that subscribe to file events from the input directory. If you enable file events on the external location that contains the files in question, you do not need to provide additional permissions when you set up the Auto Loader stream.

File notification mode with file events is more performant and scalable than directory listing. Databricks recommends file notification mode using file events instead of directory listing mode for most workloads. If you are using Auto Loader in directory listing mode today, Databricks recommends that you migrate to file notification mode using file events to see significant performance improvements. See [Configure Auto Loader streams in file notification mode.](https://docs.databricks.com/aws/en/ingestion/cloud-object-storage/auto-loader/file-notification-mode)

### Auto Loader Schema Evolution

- `addNewColumns` (default): Stream fails. New columns are added to the schema. Existing columns do not evolve data types. 
- `rescue`: Schema is never evolved and stream does not fail due to schema changes. All new columns are recorded in the rescued data column
- `failOnNewColumns`: Stream fails. Stream does not restart unless the provided schema is updated, or the offending data file is removed.
- `none`: Does not evolve the schema, new columns are ignored, and data is not rescued unless the rescuedDataColumn option is set. Stream does not fail due to schema changes.

> `addNewColumns` mode is the default when a schema is not provided, but `none` is the default when you provide a schema. `addNewColumns` is not allowed when the schema of the stream is provided, but does work if you provide your schema as a schema hint.

### Rescued Data Column

When Auto Loader infers the schema, a rescued data column is automatically added to your schema as `_rescued_data`. You can rename the column or include it in cases where you provide a schema by setting the option `rescuedDataColumn`.

The rescued data column ensures that columns that don't match with the schema are rescued instead of being dropped. The rescued data column contains any data that isn't parsed for the following reasons:

- The column is missing from the schema.
- Type mismatches.
- Case mismatches.
- The rescued data column contains a JSON containing the rescued columns and the source file path of the record.

## Auto Loader Options

### Generic Options

#### `cloudfiles.maxBytesPerTrigger`

Type: Byte String

The maximum number of new bytes to be processed in every trigger. You can specify a byte string such as `10g` to limit each microbatch to 10 GB of data. This is a soft maximum. If you have files that are 3 GB each, Databricks processes 12 GB in a microbatch. When used together with `cloudFiles.maxFilesPerTrigger`, Databricks consumes up to the lower limit of `cloudFiles.maxFilesPerTrigger` or `cloudFiles.maxBytesPerTrigger`, whichever is reached first. This option has no effect when used with `trigger.once()` (Trigger.Once() is deprecated).

**Default:** `None`

---

#### `cloudfiles.maxBytesPerTrigger`

**Type:** `Integer`

The maximum number of new files to be processed in every trigger. When used together with `cloudFiles.maxBytesPerTrigger`, Databricks consumes up to the lower limit of `cloudFiles.maxFilesPerTrigger` or `cloudFiles.maxBytesPerTrigger`, whichever is reached first. This option has no effect when used with Trigger.Once() (deprecated).

**Default:** `1000`

---

#### `cloudfiles.inferColumnTypes`

Type: Boolean

Whether to infer exact column types when leveraging schema inference. By default, columns are inferred as strings when inferring JSON and CSV datasets. See schema inference for more details.

Default: false

---

#### `ignoreCorruptFiles`

Type: `Boolean`

Whether to ignore corrupt files. If `true`, the Spark jobs will continue to run when encountering corrupted files and the contents that have been read will still be returned. Observable as `numSkippedCorruptFiles` in the `operationMetrics` column of the Delta Lake history. Available in Databricks Runtime 11.3 LTS and above.

**Default value:** `false`

---

#### `ignoreMissingFiles`

Type: `Boolean`

Whether to ignore missing files. If `true`, the Spark jobs will continue to run when encountering missing files and the contents that have been read will still be returned. Available in Databricks Runtime 11.3 LTS and above.

**Default value:** `false` for Auto Loader, `true` for `COPY INTO` (legacy)

---

#### `modifiedAfter`

Type: `Timestamp String`, for example, `2021-01-01 00:00:00.000000 UTC+0`

An optional timestamp as a filter to only ingest files that have a modification timestamp after the provided timestamp.

**Default value:** `None`

---

#### `modifiedBefore`

**Type:** `Timestamp String`, for example, `2021-01-01 00:00:00.000000 UTC+0`

An optional timestamp as a filter to only ingest files that have a modification timestamp before the provided timestamp.

**Default value:** `None`

---

### CSV Options

#### `badRecordsPath`

Type: `String`

The path to store files for recording the information about bad CSV records.

Default value: `None`

---

#### `columnNameOfCorruptRecord`

Supported for Auto Loader. Not supported for COPY INTO (legacy).

**Type:** `String`

The column for storing records that are malformed and cannot be parsed. If the mode for parsing is set as `DROPMALFORMED`, this column will be empty.

**Default value:** `_corrupt_record`

---

#### `dateFormat`

**Type:** `String`

The format for parsing date strings.

**Default value:** `yyyy-MM-dd`

---

#### `timestampFormat`

Type: `String`

The format for parsing timestamp strings.

Default value: `yyyy-MM-dd'T'HH:mm:ss[.SSS][XXX]`

---

`mode`

Type: `String`

Parser mode around handling malformed records. One of `'PERMISSIVE'`, `'DROPMALFORMED'`, and `'FAILFAST'`.

Default value: `PERMISSIVE`

#### `cloudFiles.allowOverwrites`

**Type:** `Boolean`

Whether to allow input directory file changes to overwrite existing data.

**Default:** `false`

---

#### `cloudFiles.backfillInterval`

**Type:** `Interval String`

Auto Loader can trigger asynchronous backfills at a given interval. For example `1 day` to backfill daily or `1 week` to backfill weekly.

Do not use when `cloudFiles.useManagedFileEvents` is set to `true`.

**Default:** `None`

---

#### `cloudFiles.cleanSource`

Type: `String`

Whether to **automatically delete processed files from the input directory**. When set to `OFF` (default), no files are deleted.

When set to `DELETE`, Auto Loader automatically deletes files 30 days after they are processed. To do this, Auto Loader must have write permissions to the source directory.

When set to `MOVE`, Auto Loader automatically moves files to the specified location in `cloudFiles.cleanSource.moveDestination` 30 days after they are processed. To do this, Auto Loader must have write permissions to the source directory as well as to the move location.

A file is considered processed when it has a non-null value for `commit_time` in the result of the `cloud_files_state` table valued function. The 30 day additional wait after processing can be configured using `cloudFiles.cleanSource.retentionDuration`.

> **Note:** Databricks does not recommend using this option if there are multiple streams consuming data from the source location because the fastest consumer will delete the files and they will not be ingested in the slower sources.

**Default:** OFF

---

#### `cloudFiles.cleanSource.retentionDuration`

**Type:** `Interval String`

Amount of time to wait before processed files become candidates for archival with cleanSource. Must be greater than 7 days for DELETE. No minimum restriction for MOVE.

**Default value:** `30 days`

### AWS Specific Options

#### `cloudFiles.queueUrl`

**Type:** `String`

The URL of the SQS queue. If provided, Auto Loader directly consumes events from this queue instead of setting up its own AWS SNS and SQS services.

**Default:** `None`

#### `cloudFiles.region`

---

**Type:** `String`

The region where the source S3 bucket resides and where the AWS SNS and SQS services will be created.

**Default:** The region of the EC2 instance.

`cloudFiles.queueUrl`

**Type:** `String`

The URL of the SQS queue. If provided, Auto Loader directly consumes events from this queue instead of setting up its own AWS SNS and SQS services.

**Default:** `None`

#### `pathGlobFilter` or `fileNamePattern`

Type: `String`

A potential glob pattern to provide for choosing files. Equivalent to `PATTERN` in `COPY INTO` (legacy). `fileNamePattern` can be used in `read_files`.

**Default value:** `None`

### Common Data Loading Patterns

| Patttern                 | Description                                                                                       |
|------------------------|---------------------------------------------------------------------------------------------------|
| `?`                    | Matches any single character.                                                      |
| `*`                    | Matches zero or more characters.                                                               |
| `[abc]`                | Matches a single character from character set {a, b, c}.                              |
| `[a-z]`                | Matches a single character from the character range {a…z}.                                    |
| `[^a]`                 | Matches a single character that is not from character set or range {a}. Note that the `^` character must occur immediately to the right of the opening bracket. |
| `{ab,cd}`            | Matches a string from the string set {ab, cd}.                                         |
| `{ab,c{de,fh}}`      | Matches a string from the string set {ab, cde, cfh}.                                   |

For example, if you would like to parse only `png` files in a directory that contains files with different suffixes, you can do:

In [0]:
df = spark.readStream.format("cloudFiles") \
  .option("cloudFiles.format", "binaryFile") \
  .option("pathGlobfilter", "*.png") \
  .load(base-path)